In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import r2_score

plt.style.use("seaborn-v0_8")

## Loading the data

In [2]:
apartments = pd.read_csv("../data/sales.csv")
apartments.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 538 entries, 0 to 537
Data columns (total 33 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   price_numeric               538 non-null    float64
 1   municipality                538 non-null    object 
 2   condition                   538 non-null    object 
 3   rooms                       538 non-null    float64
 4   square_m2                   538 non-null    float64
 5   equipment                   538 non-null    object 
 6   level                       538 non-null    int64  
 7   heating                     538 non-null    object 
 8   price_per_m2                538 non-null    float64
 9   closest_hospital_m          538 non-null    float64
 10  closest_clinic_m            538 non-null    float64
 11  closest_pharmacy_m          538 non-null    float64
 12  closest_school_m            538 non-null    float64
 13  closest_university_m        538 non

In [3]:
from sklearn.model_selection import train_test_split, cross_val_score

X = apartments.drop(labels=["price_numeric", "price_per_m2"], axis="columns")
y = apartments["price_numeric"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=X['municipality'])

---
## Quickly testing out different models

In [4]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.pipeline import Pipeline
numeric_features = X.select_dtypes(include=np.number).columns

non_numeric_CT = ColumnTransformer(transformers=[
  ('cat_OHE', OneHotEncoder(handle_unknown='ignore'), ["condition", "municipality", "equipment", "heating"]),
  ('numeric_transformer', "passthrough",  numeric_features)
], remainder="drop")

numeric_CT = ColumnTransformer(transformers=[
  ('cat_OHE', OneHotEncoder(handle_unknown='ignore'), ["condition", "municipality", "equipment", "heating"]),
  ('numeric_transformer', StandardScaler(), numeric_features)
], remainder="drop")

In [5]:
np.abs(y_train - y_train.median()).median()

np.float64(78513.0)

In [6]:
from sklearn.linear_model import LinearRegression

lin_reg_pipe_line = Pipeline(steps=[
  ('col_transform', numeric_CT),
  ('lin_reg', LinearRegression())
])

eval_score = cross_val_score(lin_reg_pipe_line, X_train, y_train, cv=5, scoring='neg_mean_absolute_error').mean()
lin_reg_pipe_line.fit(X_train, y_train)
test_score = lin_reg_pipe_line.score(X_test, y_test)
print(f"Evaluation score: {eval_score}, Test score: {test_score}")

Evaluation score: -74003.43034978563, Test score: -0.4274483486754148


In [7]:
from sklearn.ensemble import RandomForestRegressor
rand_forest_ensemble = Pipeline(steps=[
  ('col_transform', non_numeric_CT),
  ('rand_forest', RandomForestRegressor(random_state=42, n_estimators=100))
])

eval_score = cross_val_score(rand_forest_ensemble, X_train, y_train, cv=5, scoring='neg_mean_absolute_error').mean()
rand_forest_ensemble.fit(X_train, y_train)
test_score = rand_forest_ensemble.score(X_train, y_train)

print(f"Evaluation score: {eval_score}, Test score: {test_score}")

Evaluation score: -54706.74625581396, Test score: 0.9643779789040243


In [44]:
from sklearn.svm import SVR
svm_pipeline = Pipeline([
    ("preprocess", numeric_CT),   # includes StandardScaler for numeric
    ("svm", SVR(kernel="poly"))
])

eval_score = cross_val_score(svm_pipeline, X_train, y_train, cv=5, scoring='neg_mean_absolute_error').mean()

svm_pipeline.fit(X_train, y_train.map(np.log))
test_score = svm_pipeline.score(X_test, y_test.map(np.log))
print(f"Evaluation score: {eval_score}, Test score: {test_score}")

Evaluation score: -244541.9961620643, Test score: -62417.198581710225


In [9]:
from sklearn.linear_model import Ridge

ridge_pipeline = Pipeline(steps=[
    ("preprocess", numeric_CT),
    ("ridge", Ridge(alpha=1.0))
])

eval_score = cross_val_score(ridge_pipeline, X_train, y_train, cv=5, scoring='neg_mean_absolute_error').mean()

ridge_pipeline.fit(X_train, y_train)
test_score = ridge_pipeline.score(X_test, y_test)
print(f"Evaluation score: {eval_score}, Test score: {test_score}")

Evaluation score: -64507.93317640132, Test score: 0.3308783880060211


In [10]:
from sklearn.linear_model import SGDRegressor
sgd_pipeline = Pipeline(steps=[
  ("preprocess", numeric_CT),
  ("SGD", SGDRegressor())
])
cross_val_score(sgd_pipeline, X_train, y_train, cv=5, scoring='neg_mean_absolute_error').mean()

np.float64(-80744895.20025904)

In [11]:
from sklearn.tree import DecisionTreeRegressor
tree_pipeline = Pipeline(steps=[
  ("preprocess", numeric_CT),
  ("tree", DecisionTreeRegressor())
])
cross_val_score(tree_pipeline, X_train, y_train, cv=5, scoring='neg_mean_absolute_error').mean()

np.float64(-70599.42325581396)

In [12]:
from sklearn.neighbors import KNeighborsRegressor
knn_pipeline = Pipeline(steps=[
  ("preprocess", numeric_CT),
  ("knn", KNeighborsRegressor(weights="distance", n_neighbors=5))
])
cross_val_score(knn_pipeline, X_train, y_train, cv=5, scoring='neg_mean_absolute_error').mean()

np.float64(-61559.01471090274)

---
# Hyperparameter Tuning / Model Selection / Feature subsetting

In [45]:
from sklearn.model_selection import GridSearchCV
from sklearn.compose import TransformedTargetRegressor
from sklearn.feature_selection import SelectKBest, f_regression

model_pipeline = Pipeline(steps=[
  ("preprocess", numeric_CT),
  ('feature_selector', "passthrough"),
  ("model", LinearRegression(n_jobs=-1))
])

In [46]:
param_grid_RF = [
  {
    "model" : [RandomForestRegressor(n_jobs=-1)],
    "model__n_estimators" : [50, 100, 200, 500],
    "model__max_depth" : [None, 2, 3, 5, 10, 20, 30],
    "model__min_samples_leaf" : [1, 5, 10, 20],
    "model__min_samples_split" : [2, 10, 20],
    "model__max_features" : [None, "sqrt", "log2"]
  },
  {
    "model" : [TransformedTargetRegressor(regressor=RandomForestRegressor(n_jobs=-1), func=np.log, inverse_func=np.exp)],
    "model__regressor__n_estimators" : [50, 100, 200, 500],
    "model__regressor__max_depth" : [None, 2, 3, 5, 10, 20, 30],
    "model__regressor__min_samples_leaf" : [1, 5, 10, 20],
    "model__regressor__min_samples_split" : [2, 10, 20],
    "model__regressor__max_features" : [None, "sqrt", "log2"]
  }
]

grid_search_RF = GridSearchCV(
  estimator = model_pipeline,
  param_grid = param_grid_RF,
  n_jobs = -1,
  cv = 5,
  verbose = 1,
  scoring = 'r2',
)

grid_search_RF.fit(X_train, y_train)

Fitting 5 folds for each of 2016 candidates, totalling 10080 fits


/Users/emrearapcic-uevak/PycharmProjects/EE418-Introduction-to-Machine-Learning-Project/.venv/lib/python3.14/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/Users/emrearapcic-uevak/PycharmProjects/EE418-Introduction-to-Machine-Learning-Project/.venv/lib/python3.14/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/Users/emrearapcic-uevak/PycharmProjects/EE418-Introduction-to-Machine-Learning-Project/.venv/lib/python3.14/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.paral

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...(n_jobs=-1))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","[{'model': [RandomForestR...sor(n_jobs=-1)], 'model__max_depth': [None, 2, ...], 'model__max_features': [None, 'sqrt', ...], 'model__min_samples_leaf': [1, 5, ...], ...}, {'model': [TransformedTa...or(n_jobs=-1))], 'model__regressor__max_depth': [None, 2, ...], 'model__regressor__max_features': [None, 'sqrt', ...], 'model__regressor__min_samples_leaf': [1, 5, ...], ...}]"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'r2'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation st

In [47]:
param_grid_LinearRegression = [
  {
    "model" : [TransformedTargetRegressor(
      regressor=LinearRegression(n_jobs=-1),
      func=np.log1p,
      inverse_func=np.expm1
    ), LinearRegression(n_jobs=-1) ],
    "feature_selector" : [SelectKBest(score_func=f_regression)],
    "feature_selector__k" : [2,3,5,10,20,30, "all"]
  },
]

grid_search_LinearRegression = GridSearchCV(
  estimator = model_pipeline,
  param_grid = param_grid_LinearRegression,
  n_jobs = -1,
  cv = 5,
  verbose = 1,
  scoring = 'r2',
)

grid_search_LinearRegression.fit(X_train, y_train)

Fitting 5 folds for each of 14 candidates, totalling 70 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...(n_jobs=-1))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","[{'feature_selector': [SelectKBest(s... 0x12384dd20>)], 'feature_selector__k': [2, 3, ...], 'model': [TransformedTa...on(n_jobs=-1)), LinearRegression(n_jobs=-1)]}]"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'r2'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the com

In [48]:
param_grid_knn = [
  {
    'model' : [KNeighborsRegressor(n_jobs=-1)],
    'model__n_neighbors': [1, 3, 5, 10, 20, 30, 50, 100],
    'model__weights': ['uniform', 'distance'],
    'model__algorithm' : ['ball_tree', 'kd_tree', 'brute', 'auto'],
    'model__p' : [1, 2]
  },
  {
    'model' : [TransformedTargetRegressor(
      regressor=KNeighborsRegressor(n_jobs=-1),
      func=np.log1p,
      inverse_func=np.expm1
    ) ],
    'model__regressor__n_neighbors': [1, 3, 5, 10, 20, 30, 50, 100],
    'model__regressor__weights': ['uniform', 'distance'],
    'model__regressor__algorithm' : ['ball_tree', 'kd_tree', 'brute', 'auto'],
    'model__regressor__p' : [1, 2]
  }
]

grid_search_knn = GridSearchCV(
  estimator = model_pipeline,
  param_grid = param_grid_knn,
  n_jobs = -1,
  cv = 5,
  verbose = 1,
  scoring = 'r2',
)

grid_search_knn.fit(X_train, y_train)

Fitting 5 folds for each of 256 candidates, totalling 1280 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...(n_jobs=-1))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","[{'model': [KNeighborsRegressor(n_jobs=-1)], 'model__algorithm': ['ball_tree', 'kd_tree', ...], 'model__n_neighbors': [1, 3, ...], 'model__p': [1, 2], ...}, {'model': [TransformedTa...or(n_jobs=-1))], 'model__regressor__algorithm': ['ball_tree', 'kd_tree', ...], 'model__regressor__n_neighbors': [1, 3, ...], 'model__regressor__p': [1, 2], ...}]"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'r2'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used he

In [64]:
from sklearn.ensemble import HistGradientBoostingRegressor

param_grid_gb = [
  {
    "model": [HistGradientBoostingRegressor()],
    "model__max_depth": [3, 5, 7],
    "model__learning_rate": [0.05, 0.1],
    "model__max_iter": [200, 400],
  },
  {
    "model": [TransformedTargetRegressor(
      regressor=HistGradientBoostingRegressor(),
      func=np.log1p,
      inverse_func=np.expm1
    ) ],
    "model__regressor__max_depth": [3, 5, 7],
    "model__regressor__learning_rate": [0.05, 0.1],
    "model__regressor__max_iter": [200, 400],
  },
]

grid_search_gb = GridSearchCV(
  estimator = model_pipeline,
  param_grid = param_grid_gb,
  n_jobs = -1,
  cv = 5,
  verbose = 1,
  scoring = 'r2',
)

grid_search_gb.fit(X_train, y_train)

Fitting 5 folds for each of 24 candidates, totalling 120 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...(n_jobs=-1))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","[{'model': [HistGradientB...ingRegressor()], 'model__learning_rate': [0.05, 0.1], 'model__max_depth': [3, 5, ...], 'model__max_iter': [200, 400]}, {'model': [TransformedTa...ngRegressor())], 'model__regressor__learning_rate': [0.05, 0.1], 'model__regressor__max_depth': [3, 5, ...], 'model__regressor__max_iter': [200, 400]}]"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'r2'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged

In [52]:
param_grid_SVR = [
  {
    'model' : [SVR()],
    'model__C': [0.1, 1, 10, 100],
    'model__epsilon': [0.01, 0.1, 0.2, 0.5],
    'model__gamma': ["scale", 0.01, 0.1, 1],
    'model__kernel': ['rbf'],
  },
  {
    'model' : [SVR(kernel='linear')],
    'model__C': [0.1, 1, 10, 100],
    'model__epsilon': [0.01, 0.1, 0.2],
  },
  {
    'model' : [TransformedTargetRegressor(
      regressor=SVR(),
      func=np.log1p,
      inverse_func=np.expm1
    ) ],
    'model__regressor__C': [0.1, 1, 10, 100],
    'model__regressor__epsilon': [0.01, 0.1, 0.5],
    'model__regressor__gamma': ["scale", 0.01, 0.1, 1],
    'model__regressor__kernel': ['rbf'],
  },
  {
    'model' : [TransformedTargetRegressor(
      regressor=SVR(kernel='linear'),
      func=np.log1p,
      inverse_func=np.expm1
    ) ],

    'model__regressor__C': [0.1, 1, 10, 100],
    'model__regressor__epsilon': [0.01, 0.1, 0.2],
  }
]

grid_search_SVR = GridSearchCV(
  estimator = model_pipeline,
  param_grid = param_grid_SVR,
  n_jobs = -1,
  cv = 5,
  verbose = 1,
  scoring = 'r2',
)

grid_search_SVR.fit(X_train, y_train)

Fitting 5 folds for each of 136 candidates, totalling 680 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...(n_jobs=-1))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","[{'model': [SVR()], 'model__C': [0.1, 1, ...], 'model__epsilon': [0.01, 0.1, ...], 'model__gamma': ['scale', 0.01, ...], ...}, {'model': [SVR(kernel='linear')], 'model__C': [0.1, 1, ...], 'model__epsilon': [0.01, 0.1, ...]}, ...]"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'r2'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: intCon

In [65]:
best_models = {
  'RandomForest' : grid_search_RF.best_estimator_,
  'LinearRegression' : grid_search_LinearRegression.best_estimator_,
  'KNeighborsRegressor' : grid_search_knn.best_estimator_,
  'SVR' : grid_search_SVR.best_estimator_,
  'GradientBoostingRegressor' : grid_search_gb.best_estimator_,
}

from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error

for model in best_models.keys():
    print(f"Results for {model} are:")

    best_model = best_models[model]
    y_pred = best_model.predict(X_test)

    print("CV R²:", cross_val_score(best_model, X_train, y_train, cv=5, scoring='r2').mean())
    print("Test R²:", r2_score(y_test, y_pred))
    print("CV MAE:", np.abs(cross_val_score(best_model, X_train, y_train, cv=5, scoring='neg_mean_absolute_error').mean()))
    print("Test MAE:", mean_absolute_error(y_test, y_pred))
    print("Baseline (mean predictor) MAE:", mean_absolute_error(y_test, np.full(len(y_test), y_test.mean()))), root_mean_squared_error(y_test, np.full(len(y_test), y_test.mean()))
    print("------------")



Results for RandomForest are:
CV R²: 0.6766610555687356
Test R²: 0.7370674769900896
CV MAE: 54536.720614283615
Test MAE: 48137.75191513019
Baseline (mean predictor) MAE: 98194.83470507545
------------
Results for LinearRegression are:
CV R²: 0.6218267689189811
Test R²: -28.65960415890981
CV MAE: 61370.19018865204
Test MAE: 149255.79887634286
Baseline (mean predictor) MAE: 98194.83470507545
------------
Results for KNeighborsRegressor are:
CV R²: 0.7170706669608677
Test R²: 0.6036673483475736
CV MAE: 59054.96326225074
Test MAE: 59217.3079964562
Baseline (mean predictor) MAE: 98194.83470507545
------------
Results for SVR are:
CV R²: 0.7651588350353002
Test R²: 0.6954958305144681
CV MAE: 50574.39659855489
Test MAE: 48137.49190661944
Baseline (mean predictor) MAE: 98194.83470507545
------------
Results for GradientBoostingRegressor are:
CV R²: 0.7130743158722608
Test R²: 0.7095375509479398
CV MAE: 52041.64145918509
Test MAE: 49767.1494001483
Baseline (mean predictor) MAE: 98194.8347050754

In [70]:
import joblib

model_to_save = best_models['RandomForest'] # Pick a model to save
model_to_save.fit(X, y)
joblib.dump(model_to_save, "../models/sales_predict_ML.joblib")

['../models/sales_predict_ML.joblib']

In [71]:
loaded = joblib.load("../models/sales_predict_ML.joblib") # Check if the model saved is the right one

np.testing.assert_allclose(
    model_to_save.predict(X_test[:10]),
    loaded.predict(X_test[:10])
)